# District Map Generation for Heat Pump Suitability Visualization

## Purpose

This notebook processes raw heat pump suitability data from the FfE Open Data Portal and generates district-level CSV files with properly formatted geometry data (WKT) for visualization in the Heat Pump Potentials Germany web application.

The notebook performs the following tasks:
1. **Data Fetching**: Retrieves district-level heat pump suitability data from the FfE API
2. **GeoJSON Processing**: Downloads and processes geographic boundaries for each German state
3. **Data Transformation**: Cleans, standardizes, and merges data with geographic information
4. **Export**: Generates CSV files with WKT geometry for each of Germany's 16 federal states

## Author

Heat Pump Potentials Germany Project Team

## Date

Originally created: 2024  
Last updated: 2026-01-27

## Data Source

- **FfE Open Data Portal**: [https://opendata.ffe.de](https://opendata.ffe.de)
- **Dataset ID**: 88 (District-level heat pump suitability)
- **Data Year**: 2022

---

## Table of Contents

1. [Setup and Imports](#setup-and-imports)
   - Library imports and configuration
   - Geographic coordinate definitions

2. [Section 1: Data Fetching](#data-fetching)
   - FfE Open Data Portal API documentation
   - Dataset structure and field descriptions
   - Building type and heat source code mappings

3. [Section 2: Data Transformation](#data-transformation)
   - DataFrame conversion and column cleanup
   - Category mapping from codes to labels
   - Key transformation functions

4. [Section 3: Geometry Processing](#geometry-processing)
   - GeoJSON data source and processing
   - WKT (Well-Known Text) format explanation
   - Merging geometry with statistical data

5. [Section 4: CSV Export](#csv-export)
   - Output file format and columns
   - File naming conventions
   - Export process per state

6. [State-by-State Data Generation](#state-data)
   - [Baden-Württemberg](#baden-wuerttemberg)
   - [Bayern](#bayern)
   - [Berlin](#berlin)
   - [Brandenburg](#brandenburg)
   - [Bremen](#bremen)
   - [Hamburg](#hamburg)
   - [Hessen](#hessen)
   - [Mecklenburg-Vorpommern](#mecklenburg-vorpommern)
   - [Niedersachsen](#niedersachsen)
   - [Nordrhein-Westfalen](#nordrhein-westfalen)
   - [Rheinland-Pfalz](#rheinland-pfalz)
   - [Saarland](#saarland)
   - [Sachsen](#sachsen)
   - [Sachsen-Anhalt](#sachsen-anhalt)
   - [Schleswig-Holstein](#schleswig-holstein)
   - [Thüringen](#thueringen)

---

**Note**: Each state section follows the same pattern:
1. Load state-specific GeoJSON
2. Process and clean geometry data
3. Merge with statistical data
4. Export to CSV in `webapp/data/districts/`


---

## Section 1: Data Fetching

<a id="data-fetching"></a>

### FfE Open Data Portal API

This notebook retrieves heat pump suitability data from the **Forschungsstelle für Energiewirtschaft (FfE)** Open Data Portal. FfE is a German research organization specializing in energy economics and sustainable energy systems.

#### API Endpoint

```
https://opendata.ffe.de/api/od/v_opendata?id_opendata=eq.88
```

#### Dataset Information

| Property | Value |
|----------|-------|
| Dataset ID | 88 |
| Data Year | 2022 |
| Geographic Level | District (Landkreis/Kreisfreie Stadt) |
| Coverage | All 16 German federal states |

#### API Response Structure

The API returns a JSON array where each object represents a data point with the following fields:

| Field | Description |
|-------|-------------|
| `id_opendata` | Dataset identifier (88 for district-level heat pump data) |
| `id_region_type` | Type of geographic region |
| `region_type` | Human-readable region type name |
| `id_region` | Unique region identifier |
| `region` | District name (German: Landkreis or kreisfreie Stadt) |
| `year` | Data year (2022) |
| `internal_id` | Internal FfE identifier |
| `id_internal_1` | Building type code |
| `id_internal_2` | Heat source type code |
| `value` | Suitability proportion (0.0 to 1.0) |

#### Building Type Codes (`id_internal_1`)

| Code | Building Type |
|------|---------------|
| 1 | One- and Two-family Houses |
| 6 | Apartment Buildings (3-6 units) |
| 9 | Row Houses |
| 11 | Semi-detached Houses |
| 38 | Apartment Buildings (7+ units) |
| 100 | Total (all building types) |

#### Heat Source Codes (`id_internal_2`)

| Code | Heat Source Type |
|------|------------------|
| 0 | Total (all heat sources) |
| 1 | Air-source Heat Pump |
| 2 | Ground Probe Heat Pump |
| 3 | Ground Collector Heat Pump |
| 4 | Solar-thermal Hybrid |

#### Geographic Boundary Data (GeoJSON)

District boundaries are fetched from a separate GeoJSON source:

```
https://raw.githubusercontent.com/isellsoap/deutschlandGeoJSON/main/4_kreise/2_hoch.geo.json
```

This provides high-resolution polygon geometries for all German districts (Landkreise and kreisfreie Städte).

#### Data Fetching Function

The `fetch_data(url)` function handles API requests:
- Makes HTTP GET request to the specified URL
- Returns parsed JSON on success (HTTP 200)
- Returns `None` on failure for error handling


In [ ]:
import requests
import json
import pandas as pd
import geopandas as gpd
import os
import re
import plotly.express as px
from collections import defaultdict
from geojson import Feature, FeatureCollection
from thefuzz import fuzz, process
from shapely.geometry import mapping, shape, Polygon
from shapely import wkt

In [ ]:
pd.set_option('display.max_rows', 500)

In [ ]:
STATE_GEO_VALUES = {
    "Baden-Württemberg": (48.6, 9.1, 6.85),
    "Bayern": (48.90, 11.45, 6.3),
    "Brandenburg": (52.45, 13.1, 6.8),
    "Bremen": (53.3, 8.6, 8.6),
    "Hamburg": (53.55, 10, 9),
    "Hessen": (50.6521, 9.1624, 6.7),
    "Mecklenburg-Vorpommern": (53.8, 12.43, 7),
    "Niedersachsen": (52.64, 9.85, 7),
    "Nordrhein-Westfalen": (51.4332, 7.6616, 6.8),
    "Rheinland-Pfalz": (49.8, 7.3, 7.0),
    "Saarland": (49.3964, 6.9, 8.3),
    "Sachsen": (50.9, 13.34, 6.9),
    "Sachsen-Anhalt": (51.95, 11.69, 7),
    "Schleswig-Holstein": (54.15, 9.8, 6.8),
    "Thüringen": (50.8, 11.25, 7.3),
    "(Deutschland)": (51.1657, 10.4515, 5)
}

In [ ]:
def fuzzy_search(data_df, difference_set, threshold):
    """
    Match district names from GeoJSON to data using fuzzy string matching.

    This handles cases where district names differ slightly between datasets,
    e.g., "München, Landkreis" vs "München" or spelling variations.

    Args:
        data_df: DataFrame containing 'district' column with district names
        difference_set: Set of district names from GeoJSON not found in data_df
        threshold: Minimum similarity score (0-100) to consider a match
    """
    for e in difference_set:
        # Extract unique districts from data_df for comparison
        unique_districts = data_df['district'].unique()

        # Use rapidfuzz to find top 10 most similar district names
        # process.extract returns list of (match_string, similarity_score) tuples
        top_matches = process.extract(e, unique_districts, limit=10)

        print("----------------------------------------------------------")

        # Filter matches that meet the similarity threshold
        # A threshold of 80+ usually catches spelling variations while avoiding false matches
        valid_matches = [match for match, score in top_matches if score >= threshold]

        # Display results to help manually verify matches
        if valid_matches:
            print(f"Geo: {e}")  # GeoJSON district name
            data_str = 'Data: ' + ' '.join([f"{i+1}. {match}" for i, match in enumerate(valid_matches)])
            print(data_str)  # Matching data district names
        else:
            print(f"Kein match für: {e}")  # No match found


In [ ]:
def set_geographical_values(selected_state):
    return STATE_GEO_VALUES.get(selected_state, STATE_GEO_VALUES['(Deutschland)'])

def create_state_map(state_df, selected_state, selected_building_type, selected_heat_source):
    lat, lon, zoom = set_geographical_values(selected_state)
    
    filtered_df = state_df.query(f"building_type == {selected_building_type} & heat_source == {selected_heat_source} & federal_state == '{selected_state}'")
    
    geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"district": row.district, "district_type": row.district_type},
            "geometry": mapping(row.geometry)
        }
        for idx, row in filtered_df.iterrows()
    ]
    }
    
    fig = px.choropleth_mapbox(filtered_df, 
                            geojson=geojson, 
                            locations='district',
                            featureidkey="properties.district",
                            color='value',
                            color_continuous_scale='RdYlGn',
                            range_color=[0, 1],
                            mapbox_style="carto-darkmatter",
                            opacity=1,
                            #hover_data={'value': True},
                            hover_name=filtered_df['district'],
                            hover_data={'value': True} 
                           )

    fig.update_traces(hovertemplate='%{hovertext}<br>%{z:.1%}')
    fig.update_geos(fitbounds="locations")
    fig.update_layout(
        mapbox_zoom=zoom,
        mapbox_center={"lat": lat, "lon": lon},
        autosize=False,
        width=800, 
        height=800,
        coloraxis_showscale=False
        )
    
    return fig

In [ ]:
def remove_cities(data_df):
    return data_df[~data_df['district'].str.endswith(', Kreisfreie Stadt')]

In [ ]:
BUILDING_TYPE_MAPPING = {
        1: 'One- and Two-family Houses',
        6: 'Apartment Buildings (3-6)',
        9: 'Row Houses',
        11: 'Semi-detached Houses',
        38: 'Apartment Buildings: 7 and More Apartments',
        100: 'Total'
    }

HEAT_SOURCE_MAPPING = {
        0: 'Total',
        1: 'Air',
        2: 'Ground Probe',
        3: 'Ground Collector',
        4: 'Solar-Thermal Energy and Ice Storage'
    }

In [ ]:
DISTRICT_DATA_URL = "https://opendata.ffe.de/api/od/v_opendata?id_opendata=eq.88"
DISTRICT_GEOJSON_URL = "https://raw.githubusercontent.com/isellsoap/deutschlandGeoJSON/main/4_kreise/2_hoch.geo.json"

---

## Section 2: Data Transformation
<a id="data-transformation"></a>

### Overview

After fetching raw data from the FfE API, several transformation steps are applied to prepare the data for visualization:

1. **DataFrame Conversion**: The JSON response is converted to a pandas DataFrame
2. **Column Cleanup**: Unnecessary columns are dropped (id_opendata, id_region_type, etc.)
3. **Column Renaming**: Technical names are converted to meaningful labels:
   - `region` → `district`
   - `internal_id_1` → `building_type`
   - `internal_id_2` → `heat_source`
4. **Category Mapping**: Numeric codes are converted to human-readable labels

### Key Transformation Functions

| Function | Purpose |
|----------|----------|
| `fetch_data(url)` | Retrieves data from FfE API or GeoJSON URL |
| `fuzzy_search()` | Matches district names between datasets |
| `remove_cities()` | Filters out Kreisfreie Stadt entries |
| `set_geographical_values()` | Gets map center coordinates for each state |

### Category Mappings

The raw data uses numeric codes that need to be transformed:

**Building Type Codes** (stored in `BUILDING_TYPE_MAPPING`):
```python
{
    1: 'One- and Two-family Houses',
    6: 'Apartment Buildings (3-6)',
    9: 'Row Houses',
    11: 'Semi-detached Houses',
    12: 'Apartment Buildings (7+)',
    99: 'Total (All Building Types)'
}
```

**Heat Source Codes** (stored in `HEAT_SOURCE_MAPPING`):
```python
{
    99: 'Total (All Heat Sources)',
    1: 'Air Source',
    2: 'Ground Probe',
    3: 'Ground Collector',
    4: 'Solar-thermal'
}
```

### Data Preprocessing Flow

```
Raw JSON → DataFrame → Drop Columns → Rename Columns → Map Categories → Ready for Analysis
```

The cells below demonstrate this transformation pipeline step by step.


## Process raw data

In [ ]:
# Fetch data from API
def fetch_data(url):
    response = requests.get(url)
    if response.status_code == 200:
        return json.loads(response.text)
    else:
        return None

In [ ]:
raw_data_json = fetch_data(DISTRICT_DATA_URL)

In [ ]:
raw_data_df = pd.DataFrame(raw_data_json[0]['data']).dropna(axis=1, how='all')

In [ ]:
drop_cols = ['id_opendata', 'id_region_type', 'region_type', 'id_region', 'year', 'internal_id']
raw_data_df.drop(drop_cols, axis=1, inplace=True)

In [ ]:
raw_data_df.rename(columns={'region': 'district', 'internal_id_1': 'building_type', 'internal_id_2': 'heat_source'}, inplace=True)

---

## Section 3: Geometry Processing
<a id="geometry-processing"></a>

### Overview

Geographic boundary data is essential for visualizing heat pump suitability on maps. This section explains how WKT (Well-Known Text) geometry is processed and merged with the statistical data.

### GeoJSON Data Source

District boundaries are fetched from a publicly available GeoJSON file:
- **Source**: [isellsoap/deutschlandGeoJSON](https://github.com/isellsoap/deutschlandGeoJSON)
- **File**: `3_regierungsbezirke/4_niedrig.geo.json`
- **Coordinate System**: WGS84 (EPSG:4326)

### Processing Steps

1. **Fetch GeoJSON**: Download district boundary features
2. **Group by State**: Organize features by `NAME_1` (federal state)
3. **Export State GeoJSONs**: Save individual GeoJSON files per state to `output_geojsons/`
4. **Load with GeoPandas**: Read GeoJSON into GeoDataFrame
5. **Clean Columns**: Drop unnecessary metadata (ID_0, ISO, etc.)
6. **Rename Columns**: Standardize naming:
   - `NAME_1` → `federal_state`
   - `NAME_3` → `district`
   - `TYPE_3` → `district_type`
7. **Clean District Names**: Remove suffixes like ' Städte'
8. **Merge with Data**: Join geometry with statistical data on district name
9. **Convert to WKT**: Transform geometry column to Well-Known Text format

### WKT Format

Well-Known Text (WKT) is a text markup language for representing vector geometry objects:

```
POLYGON ((lon1 lat1, lon2 lat2, lon3 lat3, ..., lon1 lat1))
```

Example district boundary:
```
POLYGON ((9.123 48.456, 9.234 48.567, 9.345 48.456, 9.123 48.456))
```

WKT is used because:
- **Text-based**: Easy to store in CSV files
- **Standard format**: Supported by most GIS tools
- **Parseable**: Can be converted back to geometry with Shapely

### Key Functions for Geometry Processing

| Function | Purpose |
|----------|----------|
| `process_gpd_geojson()` | Cleans and standardizes GeoDataFrame columns |
| `fuzzy_search()` | Matches district names between data and GeoJSON |
| `gpd.read_file()` | Loads GeoJSON into GeoDataFrame |
| `.to_wkt()` | Converts geometry to WKT string |


## Process raw geojson

In [ ]:
raw_geojson = fetch_data(DISTRICT_GEOJSON_URL)

In [ ]:
# Initialize a dictionary to hold the features grouped by NAME_1 (federal state)
# defaultdict(list) automatically creates empty lists for new keys
geojson_by_name_1 = defaultdict(list)

# Iterate over all features in the raw GeoJSON and group them by NAME_1
# Each feature represents a district (Landkreis/Stadt) polygon
# NAME_1 property contains the federal state name (Bundesland)
for feature in raw_geojson['features']:
    name_1 = feature['properties']['NAME_1']  # Extract federal state name
    geojson_by_name_1[name_1].append(feature)  # Group feature under its state

# Create output directory for state-specific GeoJSON files
output_dir = 'output_geojsons'
os.makedirs(output_dir, exist_ok=True)  # Create if doesn't exist

# Write each state's districts to a separate GeoJSON file
# This allows loading only the relevant state data later, improving performance
for name_1, features in geojson_by_name_1.items():
    # Wrap features in a GeoJSON FeatureCollection structure
    feature_collection = FeatureCollection(features)

    # Replace spaces in state names for filesystem compatibility
    output_file_path = os.path.join(output_dir, f"{name_1.replace(' ', '_')}.geojson")

    with open(output_file_path, 'w') as f:
        json.dump(feature_collection, f)

print(f"GeoJSON created in the {output_dir} directory.")

In [ ]:
gpd_drop_cols = ['ID_0', 'ISO', 'NAME_0', 'ID_1', 'ID_2', 'NAME_2', 'ID_3', 'NL_NAME_3', 'VARNAME_3', 'ENGTYPE_3']

-----------------------------------------

---

## Section 4: CSV Export
<a id="csv-export"></a>

### Overview

After merging statistical data with geometry, the final datasets are exported as CSV files for use by the Streamlit web application.

### Output Format

Each state produces a CSV file with the following columns:

| Column | Type | Description |
|--------|------|-------------|
| `index` | Integer | Row identifier |
| `district` | String | District name (Landkreis/Stadt) |
| `building_type` | String | Building category label |
| `heat_source` | String | Heat pump type label |
| `value` | Float | Suitability proportion (0.0-1.0) |
| `federal_state` | String | German state name |
| `district_type` | String | District classification |
| `geometry` | String | WKT polygon boundary |

### Output Location

Files are saved to: `webapp/data/districts/`

**Naming Convention**: `{state_abbreviation}_full_data.csv`

| State | Filename |
|-------|----------|
| Baden-Württemberg | `BW_full_data.csv` |
| Bayern | `BY_full_data.csv` |
| Brandenburg | `BB_full_data.csv` |
| ... | ... |

### Export Process per State

For each of the 15 German states, the notebook:

1. Loads the state-specific GeoJSON
2. Processes and cleans the geometry data
3. Merges with filtered statistical data
4. Converts geometry to WKT
5. Exports to CSV with `df.to_csv()`

### Usage in Web Application

The webapp's `data_manager.py` loads these CSV files:
```python
df = pd.read_csv(f'webapp/data/districts/{state_file}')
gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df['geometry']))
```

The WKT geometry is parsed back into Shapely objects for map visualization with Plotly.


## Baden-Württemberg

In [ ]:
bw_geojson = gpd.read_file('output_geojsons/Baden-Württemberg.geojson')

### Process Geopandas Dataframe for Baden-Württemberg

In [ ]:
# Create function for geopandas processing
def process_gpd_geojson(raw_gpd):
    """
    Clean and standardize a GeoDataFrame loaded from GeoJSON.

    Steps:
    1. Remove unnecessary metadata columns from the GeoJSON
    2. Rename columns to application-standard names
    3. Clean district names (remove " Städte" suffix)
    4. Sort by district_type and district name

    Args:
        raw_gpd: GeoDataFrame loaded via gpd.read_file('filepath.geojson')

    Returns:
        Cleaned GeoDataFrame with standardized column names
    """
    # Columns to drop - these are GeoJSON metadata not needed for visualization
    # ID_0, NAME_0 = Country level; ID_1, NAME_1 = State level (keep state in rename)
    # ID_2, NAME_2 = Regierungsbezirk; ID_3, NAME_3 = District level
    gpd_drop_cols = ['ID_0', 'ISO', 'NAME_0', 'ID_1', 'ID_2', 'NAME_2', 'ID_3', 'NL_NAME_3', 'VARNAME_3', 'ENGTYPE_3']
    raw_gpd.drop(gpd_drop_cols, axis=1, inplace=True)

    # Rename to application-standard column names
    # NAME_1 = federal state, NAME_3 = district, TYPE_3 = district classification
    renamed_cols = {
        'NAME_1': 'federal_state',
        'NAME_3': 'district',
        'TYPE_3': 'district_type'
    }
    raw_gpd.rename(columns=renamed_cols, inplace=True)

    # Remove " Städte" suffix from district names for consistent matching
    # Some GeoJSON entries have "München Städte" instead of "München"
    pattern = re.compile(r' Städte')
    raw_gpd['district'] = raw_gpd['district'].apply(lambda x: re.sub(pattern, '', x))

    # Sort for consistent ordering: first by type (Landkreis/Stadt), then alphabetically
    raw_gpd = raw_gpd.sort_values(by=['district_type', 'district']).reset_index(drop=True)

    clean_gpd = raw_gpd

    return clean_gpd

In [ ]:
gpd_drop_cols = ['ID_0', 'ISO', 'NAME_0', 'ID_1', 'ID_2', 'NAME_2', 'ID_3', 'NL_NAME_3', 'VARNAME_3', 'ENGTYPE_3']

In [ ]:
bw_geojson.drop(gpd_drop_cols, axis=1, inplace=True)

In [ ]:
renamed_cols = {
    'NAME_1': 'federal_state',
    'NAME_3': 'district',
    'TYPE_3': 'district_type'
}
bw_geojson.rename(columns=renamed_cols, inplace=True)

In [ ]:
# Defining the regex pattern
pattern = re.compile(r' Städte')

In [ ]:
# Apply regex substitution
bw_geojson['district'] = bw_geojson['district'].apply(lambda x: re.sub(pattern, '', x))

In [ ]:
# Fix Heilbronn
bw_geojson.loc[bw_geojson['district'] == 'Heilbronn city', 'district'] = 'Heilbronn'

In [ ]:
# Fix Freiburg (Freiburg im Breisgau)
bw_geojson.loc[bw_geojson['district'] == 'Freiburg', 'district'] = 'Freiburg im Breisgau'

In [ ]:
# Sort
bw_geojson = bw_geojson.sort_values(by=['district_type', 'district']).reset_index(drop=True)

----------------------

#### ---DONE--- 

In [ ]:
bw_geojson

In [ ]:
bw_geojson['district_type'].value_counts()

GEODATA HAS 44 VALUES -> 35 LANDKREISE + 9 STÄDTE, KORREKT

-------------------------

#### Create raw_data subset

In [ ]:
# CREATE FUNCTION FOR FUZZY MATCH
def fuzzy_match(raw_data_df, geo_df):
    # Create copy for editing
    data_df = raw_data_df.copy()
    geo = geo_df.copy()
    
    # Initialize match values
    data_df['match'] = False
    
    # Do the fuzzy match
    for df_i, df_r in data_df.iterrows():
        for geo_i, geo_r in geo.iterrows():
            if fuzz.partial_ratio(df_r['district'], geo_r['district']) == 100:
                geo.at[df_i, 'match'] = True
                break
    
    matched_data_df = data_df
    
    return matched_data_df

In [ ]:
# Create copy for editing
bw_df = raw_data_df.copy()

In [ ]:
# Initialize match values
bw_df['match'] = False

In [ ]:
# Do the fuzzy match
for df_i, df_r in bw_df.iterrows():
    for geo_i, geo_r in bw_geojson.iterrows():
        if fuzz.partial_ratio(df_r['district'], geo_r['district']) == 100:
            bw_df.at[df_i, 'match'] = True
            break

In [ ]:
# Set up for comparison
BWDATA = pd.DataFrame(bw_df[bw_df['match'] == True]['district'].unique())
BWDATA = BWDATA.rename(columns={0: 'data'})
BWDATA = BWDATA.sort_values(by='data').reset_index(drop=True)

In [ ]:
BWGEO = pd.DataFrame(bw_geojson['district'])
BWGEO = BWGEO.rename(columns={'district': 'geo'})
BWGEO = BWGEO.sort_values(by='geo').reset_index(drop=True)

In [ ]:
# Concat for comparison
pd.concat([BWDATA, BWGEO], axis=1)

Lindau(Bodensee), Neu-Ulm belong to Bayern -> remove

In [ ]:
# Correct the flags
bw_df.loc[bw_df['district'] == 'Lindau (Bodensee)', 'match'] = False
bw_df.loc[bw_df['district'] == 'Neu-Ulm', 'match'] = False

In [ ]:
# Create the BW subset
bw_df = bw_df.loc[bw_df['match'] == True]

In [ ]:
# Remove flags
bw_df = bw_df.drop(columns='match')

In [ ]:
# Save Baden-Württemberg Data
bw_df.to_csv("BW_data.csv")

### Merge Data with Geodata

In [ ]:
bw_geo = bw_geojson.copy()

##### Start with "Kreisfreie Städte" 

In [ ]:
bw_geo_cities = bw_geo[bw_geo['district_type'] == "Kreisfreie Städte"]

In [ ]:
bw_geo_cities = bw_geo_cities.copy()
bw_geo_cities['district'] = bw_geo_cities['district'] + ', Kreisfreie Stadt'

In [ ]:
# Merge city data with city geodata
bw_merged_cities = pd.merge(bw_df, bw_geo_cities, on='district')

In [ ]:
bw_merged_cities['district'].unique()

In [ ]:
# Display all BW cities
fig = create_state_map(bw_merged_cities, 'Baden-Württemberg', 100, 1)
fig.show() # COMPLETE

#### Add Landkreise

In [ ]:
# Remove already merged cities from data
bw_data_lk = bw_df[~bw_df['district'].str.endswith(', Kreisfreie Stadt')]

In [ ]:
bw_data_lk.district.sort_values().unique()

In [ ]:
# Show geodata
bw_geo_lk = bw_geo[bw_geo['district_type'] != "Kreisfreie Städte"]

In [ ]:
# Create dictionary for mismatches: 'geo':'data' -> convert to 'data'
bw_lk_match_dict = {
    'Alb-Donau': 'Alb-Donau-Kreis',
    'Bodensee': 'Bodenseekreis',
    'Enz': 'Enzkreis',
    'Hohenlohe': 'Hohenlohekreis',
    'Main-Tauber': 'Main-Tauber-Kreis'
}

In [ ]:
bw_geo_lk

In [ ]:
# Rename districts in geoset
bw_geo_lk = bw_geo_lk.copy()
bw_geo_lk['district'] = bw_geo_lk['district'].replace(bw_lk_match_dict)

In [ ]:
bw_merged_lk = pd.merge(bw_data_lk, bw_geo_lk, on='district')

In [ ]:
# Display all BW Landkreise
fig = create_state_map(bw_merged_lk, 'Baden-Württemberg', 100, 1)
fig.show() # COMPLETE

In [ ]:
# Merge Cities with Landkreise
bw_full = pd.concat([bw_merged_cities, bw_merged_lk], axis=0)

In [ ]:
# Display full map
fig = create_state_map(bw_full, 'Baden-Württemberg', 100, 0)
fig.show()

--------------------------

<center><b>BADEN-WÜRTTEMBERG MAP IS COMPLETE</b></center>

-------------------

In [ ]:
# Export data
# bw_full.to_csv("bw_data_geo.csv")

## Bayern

In [ ]:
# Create copy for editing
by_df = raw_data_df.copy()

####  Process geodata

In [ ]:
# Read in geojson -> geopandas
by_geojson = gpd.read_file('output_geojsons/Bayern.geojson')

In [ ]:
# Process
by_geojson = process_gpd_geojson(by_geojson)

In [ ]:
# Check
by_geojson['district_type'].value_counts() # KORREKT

In [ ]:
by_geo = by_geojson.copy()
by_geo_cities = by_geo[by_geo['district_type'] == 'Kreisfreie Städte'] 

In [ ]:
# Create match dictionary
by_match_dict = {
    'Munich': 'München',
    'Nuremberg': 'Nürnberg',
    'Kempten': 'Kempten (Allgäu)', # identified later
    'Weiden': 'Weiden i.d. OPf.' # identified later
    
}
# INTEGRATED IDENTIFIED MISSING MATCHES (Kempten, Weiden)

In [ ]:
# Renaming
by_geo_cities.loc[:, 'district'] = by_geo_cities['district'].replace(by_match_dict).copy()

In [ ]:
by_geo_cities = by_geo_cities.copy()
by_geo_cities['district'] = by_geo_cities['district'] + ', Kreisfreie Stadt'

In [ ]:
by_merged_cities = pd.merge(by_df, by_geo_cities, on='district')

In [ ]:
# Display all Bayern cities
fig = create_state_map(by_merged_cities, 'Bayern', 100, 1)
fig.show()

In [ ]:
# Remove already merged cities from data
by_data_lk = remove_cities(by_df)

In [ ]:
# Show geodata
by_geo_lk = by_geo[by_geo['district_type'] != "Kreisfreie Städte"]

In [ ]:
by_geo_lk.shape # Geodaten for 71 "Landkreise"

In [ ]:
by_df

In [ ]:
# Merge data with LK
by_merged_lk = pd.merge(by_df, by_geo_lk, on='district')

In [ ]:
# Map with all "Landkreise"
fig = create_state_map(by_merged_lk, "Bayern", 100, 0)
fig.show()

In [ ]:
1860 / 30 # = 62 -> 9 Landkreise missing (71 in total)

In [ ]:
# Identify missing Landkreise
set_by_geo_lk = set(by_geo_lk['district'])
set_by_merged_lk = set(by_merged_lk['district'])

In [ ]:
set_diff_BY = set_by_geo_lk - set_by_merged_lk
set_diff_BY # Namen aus GEO welche keinen Match in DATA gefunden haben

In [ ]:
fuzzy_search(by_df, set_diff_by, 80)

In [ ]:
# Rename districts in geodata to match correct corresponding districts in the heat pump dataset
by_geo_lk.loc[by_geo_lk['district'] == 'Neumarkt', 'district'] = 'Neumarkt i.d. OPf.'
by_geo_lk.loc[by_geo_lk['district'] == 'Landsberg', 'district'] = 'Landsberg am Lech'
by_geo_lk.loc[by_geo_lk['district'] == 'Neustadt-Bad Windsheim', 'district'] = 'Neustadt a.d. Aisch-Bad Windsheim'
by_geo_lk.loc[by_geo_lk['district'] == 'Neustadt', 'district'] = 'Neustadt a.d. Waldnaab'
by_geo_lk.loc[by_geo_lk['district'] == 'Pfaffenhofen', 'district'] = 'Pfaffenhofen a.d. Ilm'
by_geo_lk.loc[by_geo_lk['district'] == 'Lindau', 'district'] = 'Lindau (Bodensee)'
by_geo_lk.loc[by_geo_lk['district'] == 'Wunsiedel', 'district'] = 'Wunsiedel i. Fichtelgebirge'
by_geo_lk.loc[by_geo_lk['district'] == 'Mühldorf', 'district'] = 'Mühldorf a. Inn'
by_geo_lk.loc[by_geo_lk['district'] == 'Dillingen', 'district'] = 'Dillingen a.d. Donau'

In [ ]:
# Update merged_lk with full geo lk data
by_merged_lk = pd.merge(by_df, by_geo_lk, on='district')

In [ ]:
# Map with all "Landkreise", complete
fig = create_state_map(by_merged_lk, "Bayern", 100, 0)
fig.show()

In [ ]:
# Merge with cities
by_full = pd.concat([by_merged_cities, by_merged_lk])

In [ ]:
# Draw full map for Bayern
fig = create_state_map(by_full, "Bayern", 100, 1)
fig.show()

-> still missing: Kempten (Allgäu) [FIXED], Weiden in der Oberpfalz [FIXED]

In [ ]:
by_df[by_df['district'].str.startswith("Kempten")][['district']].head(1)

In [ ]:
by_df[by_df['district'].str.startswith("Weiden")][['district']].head(1)

In [ ]:
# Export Bayern data
# by_full.to_csv("by_full_data.csv")

------------------------------------------------------------------------

<center><b>BAYERN MAP IS COMPLETE</b></center>

-----------------------

## Brandenburg

In [ ]:
# Create DATA copy for editing
bb_df = raw_data_df.copy()

In [ ]:
# Read in geojson -> geopandas
bb_geojson = gpd.read_file(f'output_geojsons/Brandenburg.geojson')

# Process
bb_geojson = process_gpd_geojson(bb_geojson)

# Check
bb_geojson['district_type'].value_counts() # CORRECT

-----------------------------------------------

In [ ]:
bb_geo = bb_geojson.copy()
bb_geo_cities = bb_geo[bb_geo['district_type'] == 'Kreisfreie Städte']
bb_geo_lk = bb_geo[bb_geo['district_type'] != 'Kreisfreie Städte'] 

In [ ]:
bb_geo_cities[['district']]

In [ ]:
bb_cities_set = set(bb_geo_cities['district'])
bb_lk_set = set(bb_geo_lk['district'])

In [ ]:
fuzzy_search(bb_df, bb_cities_set, 80)

In [ ]:
# Create cities match dictionary for Brandenburg
bb_cities_match_dict = {
    'Frankfurt am Oder': 'Frankfurt (Oder)',
    'Cottbus': 'Cottbus, Kreisfreie Stadt',
    'Potsdam': 'Potsdam, Kreisfreie Stadt',
    'Brandenburg an der Havel': 'Brandenburg an der Havel, Kreisfreie Stadt'
}

In [ ]:
# Rename cities w/ cities match dict
bb_geo.loc[:, 'district'] = bb_geo['district'].replace(bb_cities_match_dict)

In [ ]:
fuzzy_search(bb_df, bb_lk_set, 80) # Everything's fine -> bb_geo is ready for full merge

In [ ]:
bb_full = pd.merge(bb_df, bb_geo, on='district')

In [ ]:
fig = create_state_map(bb_full, 'Brandenburg', 100, 0)
fig.show()

-> TODO: DISPLAY TOGETHER WITH BERLIN

In [ ]:
# Export Brandenburg data
# bb_full.to_csv("bb_full_data.csv")

------------------

<center><b>BRANDENBURG MAP IS COMPLETE</b></center>

------------------------------------

## Bremen 

In [ ]:
# Create DATA copy for editing
hb_df = raw_data_df.copy()

In [ ]:
# Read in geojson -> geopandas
hb1_geojson = gpd.read_file(f'output_geojsons/Bremen_Stadt.geojson')
hb1_geojson

In [ ]:
# Process
hb1_geojson = process_gpd_geojson(hb1_geojson)

# Check
hb1_geojson['district_type'].value_counts() # BREMERHAVEN IS MISSING

In [ ]:
hb1_geojson

In [ ]:
hb1_geojson.loc[0, 'district'] = 'Bremen, Kreisfreie Stadt'

In [ ]:
hb1_geojson

In [ ]:
hb2_geojson = gpd.read_file(f'output_geojsons/Bremerhaven.geojson')
hb2_geojson = hb2_geojson[['geometry']]
bremerhaven_dict = {'federal_state': 'Bremen', 'district': 'Bremerhaven, Kreisfreie Stadt', 'district_type': 'Kreisfreie Städte'}

In [ ]:
for idx, (col_name, col_data) in enumerate(bremerhaven_dict.items()):
    hb2_geojson.insert(0, col_name, col_data)

In [ ]:
hb2_geojson

In [ ]:
hb_geojson = pd.concat([hb1_geojson, hb2_geojson], axis=0)
hb_geojson = hb_geojson.reset_index(drop=True)

In [ ]:
hb_geojson

In [ ]:
hb_df[hb_df['district'].str.startswith('Breme')]['district'].unique()

In [ ]:
hb_full = pd.merge(hb_df, hb_geojson, on='district')

In [ ]:
hb_full

In [ ]:
fig = create_state_map(hb_full, 'Bremen', 100, 0)
fig.show()

In [ ]:
#Export data
#hb_full.to_csv("hb_data_geo.csv")

## Hamburg

In [ ]:
# Create DATA copy for editing
hh_df = raw_data_df.copy()

In [ ]:
# Read in geojson -> geopandas
hh_geojson = gpd.read_file(f'output_geojsons/Hamburg.geojson')

# Process
hh_geojson = process_gpd_geojson(hh_geojson)

# Check
hh_geojson['district_type'].value_counts() # CORRECT

In [ ]:
hh_geo = hh_geojson.copy()
hh_geo

In [ ]:
hh_full = pd.merge(hh_df, hh_geo, on='district')

In [ ]:
fig = create_state_map(hh_full, 'Hamburg', 100, 0)
fig.show()

In [ ]:
# hh_full.to_csv("hh_full_data.csv")

------------------------------

<center><b>HAMBURG MAP IS COMPLETE</b></center>

------------------------------

## Hessen

In [ ]:
# Create DATA copy for editing
he_df = raw_data_df.copy()

In [ ]:
# Read in geojson -> geopandas
he_geojson = gpd.read_file(f'output_geojsons/Hessen.geojson')

# Process
he_geojson = process_gpd_geojson(he_geojson)

# Check
he_geojson['district_type'].value_counts() # should be 21 Landkreise + 5 Kreisfreie Städte

In [ ]:
he_geo = he_geojson.copy()
he_geo_cities = he_geo[he_geo['district_type'] == 'Kreisfreie Städte']
he_geo_lk = he_geo[he_geo['district_type'] != 'Kreisfreie Städte'] 

In [ ]:
he_geo_cities

In [ ]:
he_geo.loc[he_geo['district'] == 'Main-Taunus-Kreis', 'district_type'] = 'Landkreise'

In [ ]:
he_geo_cities = he_geo[he_geo['district_type'] == 'Kreisfreie Städte']
he_geo_lk = he_geo[he_geo['district_type'] != 'Kreisfreie Städte'] 

In [ ]:
he_geo.district_type.value_counts() # CORRECT

In [ ]:
fuzzy_search(he_df, set(he_geo_cities['district']), 80)

In [ ]:
he_cities_match_dict = {
    'Offenbach am Main': 'Offenbach am Main, Kreisfreie Stadt',
    'Frankfurt am Main': 'Frankfurt am Main, Kreisfreie Stadt',
    'Darmstadt': 'Darmstadt, Kreisfreie Stadt',
    'Wiesbaden': 'Wiesbaden, Kreisfreie Stadt',
    'Kassel': 'Kassel, Kreisfreie Stadt'
}

In [ ]:
# Rename cities w/ cities match dict
he_geo_cities.loc[:, 'district'] = he_geo_cities['district'].replace(he_cities_match_dict)

In [ ]:
he_merged_cities = pd.merge(he_df, he_geo_cities, on='district')

In [ ]:
fig = create_state_map(he_merged_cities, 'Hessen', 100, 0)
fig.show()

In [ ]:
he_geo_lk.shape

In [ ]:
fuzzy_search(he_df, set(he_geo_lk['district']), 80)

In [ ]:
he_merged_lk

In [ ]:
he_merged_lk = pd.merge(he_df, he_geo_lk, on='district')

In [ ]:
fig = create_state_map(he_merged_lk, 'Hessen', 100, 0)
fig.show()

In [ ]:
he_full = pd.concat([he_merged_cities, he_merged_lk])

In [ ]:
fig = create_state_map(he_full, 'Hessen', 100, 0)
fig.show()

In [ ]:
# he_full.to_csv('he_full_data.csv')

------------------------------------

<center><b>HESSEN MAP IS COMPLETE</b></center>

---------------------------

## Mecklenburg-Vorpommern

In [ ]:
# Create DATA copy for editing
mv_df = raw_data_df.copy()

In [ ]:
# Read in geojson -> geopandas
mv_geojson = gpd.read_file(f'output_geojsons/Mecklenburg-Vorpommern.geojson')

# Process
#mv_geojson = process_gpd_geojson(mv_geojson)

# Check
#mv_geojson['district_type'].value_counts() # Sollten 6 Landkreise und 2 Städte sein...

In [ ]:
mv_geojson

In [ ]:
mv_full

In [ ]:
mv_geojson = mv_geojson[['krs_name', 'lan_name', 'krs_type', 'geometry']]
mv_geojson['krs_name'] = mv_geojson['krs_name'].str[0]
mv_geojson['lan_name'] = mv_geojson['lan_name'].str[0]
mv_geojson['krs_type'] = mv_geojson['krs_type'].str[0]
mv_geojson

In [ ]:
mv_geojson = mv_geojson.rename(columns={'krs_name': 'district', 'krs_type': 'district_type', 'lan_name': 'federal_state'})

In [ ]:
fuzzy_search(mv_df, set(mv_geojson['district']), 70)

In [ ]:
mv_match_dict = {
    'Landkreis Vorpommern-Rügen': 'Vorpommern-Rügen',
    'Kreisfreie Stadt Rostock': 'Rostock, Kreisfreie Stadt',
    'Kreisfreie Stadt Schwerin': 'Schwerin, Kreisfreie Stadt',
    'Landkreis Mecklenburgische Seenplatte': 'Mecklenburgische Seenplatte',
    'Landkreis Vorpommern-Greifswald': 'Vorpommern-Greifswald',
    'Landkreis Nordwestmecklenburg': 'Nordwestmecklenburg',
    'Landkreis Rostock': 'Rostock',
    'Landkreis Ludwigslust-Parchim': 'Ludwigslust-Parchim'
}

In [ ]:
mv_geo = mv_geojson.copy()
mv_geo.loc[:, 'district'] = mv_geo['district'].replace(mv_match_dict)

In [ ]:
mv_full = pd.merge(mv_df, mv_geo, on='district')

In [ ]:
fig = create_state_map(mv_full, 'Mecklenburg-Vorpommern', 100, 0)
fig.show()

In [ ]:
# Export MV
#mv_full.to_csv("mv_full_data.csv")

## Niedersachsen

In [ ]:
# Create DATA copy for editing
ni_df = raw_data_df.copy()

In [ ]:
# Read in geojson -> geopandas
ni_geojson = gpd.read_file(f'output_geojsons/Niedersachsen.geojson')

# Process
#ni_geojson = process_gpd_geojson(ni_geojson)

# Check
#ni_geojson['district_type'].value_counts() # 37LK + 8KfS

In [ ]:
ni_geojson = ni_geojson[['krs_name', 'lan_name', 'krs_type', 'geometry']]
ni_geojson['krs_name'] = ni_geojson['krs_name'].str[0]
ni_geojson['lan_name'] = ni_geojson['lan_name'].str[0]
ni_geojson['krs_type'] = ni_geojson['krs_type'].str[0]
ni_geojson

In [ ]:
ni_geojson = ni_geojson.rename(columns={'krs_name': 'district', 'krs_type': 'district_type', 'lan_name': 'federal_state'})

In [ ]:
fuzzy_search(ni_df, set(ni_geojson['district']), 70)

In [ ]:
# Naming discrepencies are consistent, no dict necessary -> regex
ni_geojson['district'] = ni_geojson['district'].str.replace(r'^Kreisfreie Stadt (.+)$', r'\1, Kreisfreie Stadt', regex=True)
ni_geojson['district'] = ni_geojson['district'].str.replace(r'^Landkreis ', '', regex=True)

In [ ]:
ni_geojson

In [ ]:
ni_full = pd.merge(ni_df, ni_geojson, on='district')

In [ ]:
fig = create_state_map(ni_full, 'Niedersachsen', 100, 0)
fig.show()

In [ ]:
# Export
# ni_full.to_csv('ni_full_data.csv')

## Nordrhein-Westfalen

In [ ]:
# Create DATA copy for editing
nw_df = raw_data_df.copy()

In [ ]:
# Read in geojson -> geopandas
nw_geojson = gpd.read_file(f'output_geojsons/Nordrhein-Westfalen.geojson')

# Process
nw_geojson = process_gpd_geojson(nw_geojson)

# Check
nw_geojson['district_type'].value_counts() # 31 LK + 22 S 

In [ ]:
nw_geojson[nw_geojson['district_type'] == 'Kreisfreie Städte']

In [ ]:
nw_geojson[nw_geojson['district_type'] != 'Kreisfreie Städte']

Sonderfall Aachen: GEO: LK + S, DATA: Städteregion Aachen

In [ ]:
fuzzy_search(nw_df, set(nw_geojson['district']), 80)

In [ ]:
nw_match_dict = {
    'Leverkusen': 'Leverkusen, Kreisfreie Stadt',
    'Cleves': 'Kleve',
    'Essen': 'Essen, Kreisfreie Stadt',
    'Krefeld': 'Krefeld, Kreisfreie Stadt',
    'Düsseldorf': 'Düsseldorf, Kreisfreie Stadt',
    'Cologne': 'Köln, Kreisfreie Stadt',
    'Duisburg': 'Duisburg, Kreisfreie Stadt',
    'Dortmund': 'Dortmund, Kreisfreie Stadt',
    'Gelsenkirchen': 'Gelsenkirchen, Kreisfreie Stadt',
    'Bottrop': 'Bottrop, Kreisfreie Stadt',
    'Bochum': 'Bochum, Kreisfreie Stadt',
    'Münster': 'Münster, Kreisfreie Stadt',
    'Mülheim': 'Mülheim an der Ruhr, Kreisfreie Stadt',
    'Solingen': 'Solingen, Kreisfreie Stadt',
    'Wuppertal': 'Wuppertal, Kreisfreie Stadt',
    'Mönchengladbach': 'Mönchengladbach, Kreisfreie Stadt',
    'Ennepe-Ruhr': 'Ennepe-Ruhr-Kreis',
    'Hamm': 'Hamm, Kreisfreie Stadt',
    'Herne': 'Herne, Kreisfreie Stadt',
    'Oberhausen': 'Oberhausen, Kreisfreie Stadt',
    'Bielefeld': 'Bielefeld, Kreisfreie Stadt',
    'Hagen': 'Hagen, Kreisfreie Stadt',
    'Rhein-Sieg': 'Rhein-Sieg-Kreis',
    'Remscheid': 'Remscheid, Kreisfreie Stadt',
    'Bonn': 'Bonn, Kreisfreie Stadt'
}

In [ ]:
nw_geo = nw_geojson.copy()
nw_geo.loc[:, 'district'] = nw_geo['district'].replace(nw_match_dict)

In [ ]:
nw_full_without_aachen = pd.merge(nw_df, nw_geo, on='district')

In [ ]:
fig = create_state_map(nw_full_without_aachen, 'Nordrhein-Westfalen', 100, 0)
fig.show()

In [ ]:
# Add Aachen, use same DATA for differing GEO
aachen_geo_city = nw_geo[nw_geo['district'] == 'Aachen'].iloc[[0]]
aachen_geo_lk = nw_geo[nw_geo['district'] == 'Aachen'].iloc[[1]]

In [ ]:
aachen_data = nw_df[nw_df['district'].str.contains('Aachen')]

In [ ]:
aachen_geo_lk['district'] = 'Städteregion Aachen'
aachen_geo_city['district'] = 'Städteregion Aachen'

In [ ]:
# Merge with Aachen LK GEO
aachen_merged_lk = pd.merge(nw_df, aachen_geo_lk, on='district')

In [ ]:
# Merge with Aachen City GEO
aachen_merged_city = pd.merge(nw_df, aachen_geo_city, on='district')

In [ ]:
# Rename city region of Aachen
aachen_merged_city.loc[:, 'district'] = 'Aachen (Stadt)'

In [ ]:
# Combine NRW data with Aachen
nw_full = pd.concat([nw_full_without_aachen, aachen_merged_lk, aachen_merged_city])

In [ ]:
# Show complete NRW map
fig = create_state_map(nw_full, 'Nordrhein-Westfalen', 100, 0)
fig.show()

In [ ]:
# Export full NRW-Data
# nw_full.to_csv("nw_full_data.csv")

---------------------------------

<center><b>NORDRHEIN-WESTFALEN MAP COMPLETE</b></center>

-----------------------

## Rheinland-Pfalz 

In [ ]:
# Create DATA copy for editing
rp_df = raw_data_df.copy()

In [ ]:
# Read in geojson -> geopandas
rp_geojson = gpd.read_file(f'output_geojsons/Rheinland-Pfalz.geojson')

# Process
rp_geojson = process_gpd_geojson(rp_geojson)

# Check
rp_geojson['district_type'].value_counts() # 24 LK + 12 S

In [ ]:
rp_geo = rp_geojson.copy()

In [ ]:
rp_geo[rp_geo['district_type'] == 'Kreisfreie Städte']

--> Pirmasens is <b>missing</b> from GEO

In [ ]:
rp_geo

Datasource for NUTS: https://ec.europa.eu/eurostat/web/gisco/geodata/reference-data/administrative-units-statistical-units/nuts

In [ ]:
# Read in NUTS geojson file into geopandas
nuts = gpd.read_file("output_geojsons/NUTS_RG_01M_2021_4326.geojson")

In [ ]:
# Get Pirmasens GEO
pirmasens_geo = nuts[nuts['NUTS_ID'] == 'DEB37']
pirmasens_geo = pirmasens_geo.rename(columns={'NAME_LATN': 'district'})
pirmasens_geo = pirmasens_geo.loc[:, ['district', 'geometry']]

In [ ]:
# Add columns to fit format
pirmasens_geo.insert(0, 'federal_state', 'Rheinland-Pfalz')
pirmasens_geo.insert(2, 'district_type', 'Kreisfreie Städte')

In [ ]:
rp_geo.crs

In [ ]:
pirmasens_geo.crs

In [ ]:
# Different Coordinate Reference Systems (CRS) -> convert before concat
# pirmasens_geo = pirmasens_geo.to_crs(rp_geo.crs)

In [ ]:
# Add Pirmasens to GEO
rp_geo = pd.concat([pirmasens_geo, rp_geo])

In [ ]:
rp_geo = rp_geo.reset_index(drop=True)

In [ ]:
# Do the fuzzy match
fuzzy_search(rp_df, set(rp_geo['district']), 80)

In [ ]:
# Create dictionaries for string transformation
rp_cities_match_dict = {
    'Trier': 'Trier, Kreisfreie Stadt',
    'Zweibrücken': 'Zweibrücken, Kreisfreie Stadt',
    'Landau': 'Landau in der Pfalz, Kreisfreie Stadt',
    'Speyer Spires': 'Speyer, Kreisfreie Stadt',
    'Ludwigshafen': 'Ludwigshafen am Rhein, Kreisfreie Stadt',
    'Koblenz Coblenz': 'Koblenz, Kreisfreie Stadt',
    'Neustadt': 'Neustadt an der Weinstraße, Kreisfreie Stadt',
    'Mainz': 'Mainz, Kreisfreie Stadt',
    'Frankenthal': 'Frankenthal (Pfalz), Kreisfreie Stadt',
    'Worms': 'Worms, Kreisfreie Stadt',
    'Kaiserslautern': 'Kaiserslautern, Kreisfreie Stadt'
}

rp_lk_match_dict = {
    'Altenkirchen': 'Altenkirchen (Westerwald)',
    'Rhein-Lahn': 'Rhein-Lahn-Kreis',
    'Bitburg-Prüm': 'Eifelkreis Bitburg-Prüm',
    'Rhein-Hunsrück': 'Rhein-Hunsrück-Kreis'
}

In [ ]:
rp_geo_cities = rp_geo.loc[rp_geo['district_type'] == 'Kreisfreie Städte'].copy()
rp_geo_cities.loc[:, 'district'] = rp_geo_cities['district'].replace(rp_cities_match_dict)
rp_geo_cities

In [ ]:
rp_merged_cities = pd.merge(rp_df, rp_geo_cities, on='district')

In [ ]:
fig = create_state_map(rp_merged_cities, 'Rheinland-Pfalz', 100, 0)
fig.show()

In [ ]:
rp_geo_lk = rp_geo.loc[rp_geo['district_type'] != 'Kreisfreie Städte'].copy()
rp_geo_lk.loc[:, 'district'] = rp_geo_lk['district'].replace(rp_lk_match_dict)
rp_geo_lk

In [ ]:
rp_merged_lk = pd.merge(rp_df, rp_geo_lk, on='district')

In [ ]:
fig = create_state_map(rp_merged_lk, 'Rheinland-Pfalz', 100, 0)
fig.show()

In [ ]:
rp_full = pd.concat([rp_merged_cities, rp_merged_lk])

In [ ]:
fig = create_state_map(rp_full, 'Rheinland-Pfalz', 100, 0)
fig.show()

In [ ]:
# Fix Südwestpfalz

# Get corect Südwestpfalz GEO
suedwestpfalz_geo = nuts[nuts['NUTS_ID'] == 'DEB3K']
suedwestpfalz_geo = suedwestpfalz_geo.rename(columns={'NAME_LATN': 'district'})
suedwestpfalz_geo = suedwestpfalz_geo.loc[:, ['district', 'geometry']]

# Add columns to fit format
suedwestpfalz_geo.insert(0, 'federal_state', 'Rheinland-Pfalz')
suedwestpfalz_geo.insert(2, 'district_type', 'Landkreise')

# Different Coordinate Reference Systems (CRS) -> convert before concat
suedwestpfalz_geo = suedwestpfalz_geo.to_crs(rp_geo.crs).reset_index(drop=True)
suedwestpfalz_geo

In [ ]:
# Update Südwestpfalz geometry in rp_full
rp_full.loc[rp_full['district'] == 'Südwestpfalz', 'geometry'] = suedwestpfalz_geo.at[0, 'geometry']

In [ ]:
rp_full[rp_full['district'] == 'Südwestpfalz']

In [ ]:
fig = create_state_map(rp_full, 'Rheinland-Pfalz', 100, 0)
fig.show()

In [ ]:
# Export rp_full
# rp_full.to_csv("rp_data_full.csv")

TODO: Fix Südwestpfalz using shapely

--------------------

<center><b>RHEINLAND-PFALZ MAP (ALMOST) COMPLETE</b></center>

---------------------

## Saarland 

In [ ]:
# Create DATA copy for editing
sl_df = raw_data_df.copy()

In [ ]:
# Read in geojson -> geopandas
sl_geojson = gpd.read_file(f'output_geojsons/Saarland.geojson')

# Process
sl_geojson = process_gpd_geojson(sl_geojson)

# Check
sl_geojson['district_type'].value_counts()

In [ ]:
sl_geo = sl_geojson.copy()

In [ ]:
fuzzy_search(sl_df, set(sl_geo['district']), 80)

In [ ]:
sl_match_dict = {
    'Saarpfalz': 'Saarpfalz-Kreis',
    'Saarbrücken': 'Regionalverband Saarbrücken',
    'Sankt Wendel': 'St. Wendel'
}

In [ ]:
sl_geo.loc[:, 'district'] = sl_geo['district'].replace(sl_match_dict)

In [ ]:
sl_full = pd.merge(sl_df, sl_geo, on='district')

In [ ]:
fig = create_state_map(sl_full, 'Saarland', 100, 0)
fig.show()

In [ ]:
# Export
# sl_full.to_csv('sl_full_data.csv')

--------------------

<center><b>SAARLAND MAP COMPLETE</b></center>

---------------------

## Sachsen

In [ ]:
sn_df = raw_data_df.copy()

In [ ]:
sn_geojson = gpd.read_file("output_geojsons/Sachsen.geojson")

In [ ]:
sn_geojson = sn_geojson[['krs_name', 'lan_name', 'krs_type', 'geometry']]
sn_geojson['krs_name'] = sn_geojson['krs_name'].str[0]
sn_geojson['lan_name'] = sn_geojson['lan_name'].str[0]
sn_geojson['krs_type'] = sn_geojson['krs_type'].str[0]
sn_geojson

In [ ]:
sn_geojson = sn_geojson.rename(columns={'krs_name': 'district', 'krs_type': 'district_type', 'lan_name': 'federal_state'})

In [ ]:
fuzzy_search(sn_df, set(sn_geojson['district']), 80)

In [ ]:
sn_geojson['district'] = sn_geojson['district'].str.replace(r'^Landkreis ', '', regex=True)
sn_geojson['district'] = sn_geojson['district'].str.replace(r'^Kreisfreie Stadt ', '', regex=True)

In [ ]:
sn_full = pd.merge(sn_df, sn_geojson, on='district')

In [ ]:
fig = create_state_map(sn_full, 'Sachsen', 100, 0)
fig.show()

In [ ]:
leipzig_geo = sn_full.loc[sn_full['district'] == 'Leipzig']
leipzig_geo_array = leipzig_geo['geometry'].unique()

In [ ]:
leipzig_geo_array

In [ ]:
leipzig_geo_series = gpd.GeoSeries(leipzig_geo_array)

In [ ]:
leipzig_geo_united = leipzig_geo_series.unary_union

In [ ]:
leipzig_geo_united

In [ ]:
# Update all Geodata with merged Leipzig (city + district)
sn_full.loc[sn_full['district'] == 'Leipzig', 'geometry'] = leipzig_geo_united

In [ ]:
fig = create_state_map(sn_full, 'Sachsen', 100, 0)
fig.show()

In [ ]:
# Export
# sn_full.to_csv("sn_full_data.csv")

## Sachsen-Anhalt

In [ ]:
st_df = raw_data_df.copy()
st_geojson = gpd.read_file("output_geojsons/Sachsen-Anhalt.geojson")
# st_geo = process_gpd_geojson(st_geo)

In [ ]:
st_geojson

In [ ]:
## 11 LK + 3 S (Kreisreform 2007)

In [ ]:
st_geojson = st_geojson[['krs_name', 'lan_name', 'krs_type', 'geometry']]
st_geojson['krs_name'] = st_geojson['krs_name'].str[0]
st_geojson['lan_name'] = st_geojson['lan_name'].str[0]
st_geojson['krs_type'] = st_geojson['krs_type'].str[0]
st_geojson = st_geojson.rename(columns={'krs_name': 'district', 'krs_type': 'district_type', 'lan_name': 'federal_state'})
st_geojson

In [ ]:
fuzzy_search(st_df, set(st_geojson['district']), 80)

In [ ]:
# Naming discrepencies are consistent, no dict necessary -> regex
st_geojson['district'] = st_geojson['district'].str.replace(r'^Kreisfreie Stadt (.+)$', r'\1, Kreisfreie Stadt', regex=True)
st_geojson['district'] = st_geojson['district'].str.replace(r'^Landkreis ', '', regex=True)

In [ ]:
st_full = pd.merge(st_df, st_geojson, on='district')

In [ ]:
fig = create_state_map(st_full, 'Sachsen-Anhalt', 100, 0)
fig.show()

In [ ]:
st_full.to_csv("st_full_data.csv")

## Schleswig-Holstein

In [ ]:
sh_df = raw_data_df.copy()
sh_geo = gpd.read_file("output_geojsons/Schleswig-Holstein.geojson")
sh_geo = process_gpd_geojson(sh_geo)

In [ ]:
# Check
sh_geo["district_type"].value_counts() # 11 + 4 KORREKT

In [ ]:
fuzzy_search(sh_df, set(sh_geo.district), 80)

In [ ]:
sh_match_dict = {
    "Lübeck": "Lübeck, Kreisfreie Stadt",
    "Neumünster": "Neumünster, Kreisfreie Stadt",
    "Kiel": "Kiel, Kreisfreie Stadt",
    "Lauenburg": "Herzogtum Lauenburg",
    "Flensburg": "Flensburg, Kreisfreie Stadt"
}

In [ ]:
sh_geo.district = sh_geo.district.replace(sh_match_dict)

In [ ]:
sh_merged = sh_df.merge(sh_geo, on="district")

In [ ]:
sh_merged.district.nunique()

In [ ]:
fig = create_state_map(sh_merged, "Schleswig-Holstein", 100, 0)
fig.show()

In [ ]:
# Export
# sh_merged.to_csv("sh_full_data.csv")

-------------------

<center><b> SCHLESWIG-HOLSTEIN MAP IS COMPLETE</center></b>

---------------------

## Thüringen

In [ ]:
th_df = raw_data_df.copy()
th_geo = gpd.read_file("output_geojsons/Thüringen.geojson")
th_geo = process_gpd_geojson(th_geo)

In [ ]:
th_geo.district_type.value_counts() # 17 + 5 KORREKT

In [ ]:
fuzzy_search(th_df, set(th_geo.district), 80)

In [ ]:
th_match_dict = {
    "Weimar": "Weimar, Kreisfreie Stadt",
    "Gera": "Gera, Kreisfreie Stadt",
    "Saale-Holzland": "Saale-Holzland-Kreis",
    "Jena": "Jena, Kreisfreie Stadt",
    "Saale-Orla": "Saale-Orla-Kreis",
    "Unstrut-Hainich": "Unstrut-Hainich-Kreis",
    "Erfurt": "Erfurt, Kreisfreie Stadt",
    "Suhl": "Suhl, Kreisfreie Stadt"
}

In [ ]:
th_geo.district = th_geo.district.replace(th_match_dict)

In [ ]:
th_merge = th_df.merge(th_geo, on="district")

In [ ]:
fig = create_state_map(th_merge, "Thüringen", 100, 0)
fig.show()

In [ ]:
# th_merge.to_csv("th_full_data.csv")